In [3]:
import numpy as np
import re

In [4]:
import os

for root, dirs, files in os.walk('.', topdown=True):
    for name in files:
        if 'text8' in name:
            print(os.path.join(root, name))

.\text8


In [5]:
doc = open('text8','r').read().lower()

In [6]:
len(doc)

100000000

In [8]:
import re

corpus_list = re.split(r'\W+', doc)

print("Total tokens:", len(corpus_list))
print("Sample:", corpus_list[:10])

Total tokens: 17005208
Sample: ['', 'anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used']


In [9]:
len(corpus_list)

17005208

In [10]:
cutOffValue = 100
from collections import defaultdict
frequency = defaultdict(int)
for token in corpus_list:
    frequency[token] += 1
processedCorpus_list = [token for token in corpus_list 
                        if frequency[token] >= cutOffValue]

In [11]:
len(processedCorpus_list)

15471435

In [12]:
allWords = np.array(list(frequency.keys()))
allCounts = np.array(list(frequency.values()))

In [13]:
vocab = allWords[allCounts >= cutOffValue]
wordCounts = allCounts[allCounts >= cutOffValue]

In [14]:
len(vocab)

11815

In [15]:
from scipy.sparse import lil_matrix

In [16]:
def computeWordContextMatrix(corpus_list,vocab=None,windowSize=2):
    if vocab is None:
        vocab = sorted(list(set(cospus_list)))
    numWords = len(vocab)
    M = np.zeros((numWords,numWords))
    #M = lil_matrix((numWords,numWords))
    W2I = dict(zip(vocab,np.arange(numWords)))
    I2W = dict(zip(np.arange(numWords),vocab))
    doc = corpus_list
    curIdx = 0
    docLen = len(doc)
    while curIdx < docLen:
        left = max(curIdx-windowSize,0)
        right = min(curIdx+windowSize+1,docLen)
        wordsInContext = doc[left:curIdx] + doc[curIdx+1:right]
        currentWord = doc[curIdx]
        currentWordIdx = W2I[currentWord]
        for word in wordsInContext:
            contextWordIdx = W2I[word]
            M[currentWordIdx,contextWordIdx] += 1
        curIdx += 1
    return M,W2I,I2W

In [17]:
M,W2I,I2W = computeWordContextMatrix(processedCorpus_list,vocab)

In [18]:
M.shape

(11815, 11815)

In [19]:
len(vocab)

11815

In [20]:
word = 'good'
print(W2I[word],I2W[190])

190 good


In [21]:
v = M[W2I['good'],:]
print(v)

[  0.   0. 214. ...   0.   0.   0.]


In [22]:
v.shape

(11815,)

In [25]:
def pmi(M, positive=True):
    col_totals = np.sum(M,axis=0)
    total = col_totals.sum()
    row_totals = np.sum(M,axis=1)
    expected = np.outer(row_totals, col_totals) / total
    M = M / expected    
    with np.errstate(divide='ignore'):
        M = np.log(M)
    M[np.isinf(M)] = 0.0  
    if positive:
        M[M < 0] = 0.0
    return M

In [26]:
M = pmi(M)

In [27]:
M[W2I['good'],:]

array([0.       , 0.       , 0.4605286, ..., 0.       , 0.       ,
       0.       ])

In [28]:
from sklearn.decomposition import TruncatedSVD,PCA,IncrementalPCA

In [29]:
transformer = TruncatedSVD(n_components=100)

In [30]:
M_reduced = transformer.fit_transform(M)

In [31]:
M_reduced.shape

(11815, 100)

In [32]:
M.shape

(11815, 11815)

In [33]:
M_reduced[W2I['good'],:]

array([23.58455957, -3.84230766, -3.72714356, -4.18267387, -4.50620884,
        7.88498299, -1.00250276, -3.38180168,  0.05665934, -2.34167378,
        7.71499638,  2.24904646, -2.86826339,  3.30221606,  0.53320598,
        0.14635419, -6.92015815,  4.46841084, -0.8760377 , -0.34353128,
        1.34133774,  0.79350043,  1.88268897,  3.98012621,  2.32949115,
        0.84257066, -1.32400431,  2.26281258,  1.3933175 ,  0.1527749 ,
        3.98517   ,  3.96093489, -0.49099824,  4.49648444,  0.74789848,
        1.37857294,  1.09856087, -3.20204068,  1.81735977, -2.53039805,
       -0.31617875, -1.94760714, -1.31147593,  2.45661826, -1.06576762,
       -2.0124504 ,  1.34765152,  1.64064949,  0.33875526,  1.48831567,
        3.46017976, -1.11376309, -0.6368045 ,  2.40226351,  0.21263676,
       -2.01127262, -0.51688105, -0.32532872, -2.73119361, -2.07347804,
       -0.30711064, -3.68348563,  0.4042296 ,  1.52522378,  0.89293842,
       -0.10152048, -1.09013733,  1.27131881, -0.94102225, -0.50

In [34]:
def getNorms(E):
    if E.ndim == 1:
        E = E[np.newaxis,:]
    nrms = np.sum(E**2,axis=1)**0.5
    return nrms

In [35]:
def normalize(E):
    if E.ndim == 1:
        E = E[np.newaxis,:]
    nrms = getNorms(E)
    return E/nrms[:,np.newaxis]

In [36]:
def cosineSimilarity(E,v):
    E = normalize(E)
    v = normalize(v)
    scores = E.dot(v.T)
    return scores

In [37]:
def getMostSimilarWords(E,word,W2I,topn=10):
    if type(word) is str:
        v = E[W2I[word],:]
    else:
        v = word
    scores = cosineSimilarity(E,v)
    scores = scores.squeeze()
    sortedScores = np.sort(scores)[::-1]
    idx = np.argsort(scores)[::-1]
    topNScores = sortedScores[:topn]
    topNWordsIdx = idx[:topn]
    return topNScores,topNWordsIdx

In [38]:
scores,idx = getMostSimilarWords(M_reduced,'exceptional',W2I)
for i in range(len(idx)):
    print(scores[i],I2W[idx[i]])

1.0 exceptional
0.8007383274656908 extraordinary
0.7919928827778586 remarkable
0.7743547647555312 subtle
0.7560221631867516 unusual
0.7529383018830899 striking
0.7413356977637728 curious
0.7247146014015986 inherent
0.7195728614552799 possessing
0.7072525630964626 noticeable


In [39]:
print(getNorms(M_reduced[[1,4,5],:]))

[20.45092653 31.58714079 34.45926518]


In [40]:
v.shape

(11815,)

In [41]:
import dill

In [42]:
def saveModel(model,fileNameDotpkl):
    with open(fileNameDotpkl,'wb') as f:
        dill.dump(model,f)
def loadModel(fileNameDotpkl):
    with open(fileNameDotpkl,'rb') as f:
        return dill.load(f)

In [47]:
from gensim.models import Word2Vec
import dill

In [48]:
with open('text8', 'r') as f:
    text = f.read().lower().split()

In [49]:
model = Word2Vec(sentences=[text], vector_size=100, window=5, min_count=5, workers=4)

In [50]:
E = model.wv.vectors
W2I = {word: i for i, word in enumerate(model.wv.index_to_key)}
I2W = {i: word for i, word in enumerate(model.wv.index_to_key)}
vocab = list(model.wv.index_to_key)

In [52]:
def saveModel(fileNameDotpkl, E, W2I, I2W, vocab):
    with open(fileNameDotpkl, 'wb') as f:
        dill.dump((E, W2I, I2W, vocab), f)

saveModel('w2v_100d.pkl', E, W2I, I2W, vocab)
print("Model saved successfully!")

Model saved successfully!


In [53]:
def loadModel(fileNameDotpkl):
    with open(fileNameDotpkl, 'rb') as f:
        return dill.load(f)

E, W2I, I2W, vocab = loadModel('w2v_100d.pkl')
print("Model loaded successfully!")

Model loaded successfully!


In [54]:
saveModel('w2v_100d.pkl', E, W2I, I2W, vocab)

In [55]:
E,W2I,I2W,vocab = loadModel('w2v_100d.pkl')

In [56]:
E.shape

(71290, 100)

In [57]:
E[W2I['good'],:]

array([ 0.00148202,  0.00265593,  0.00772842, -0.0050383 , -0.00720587,
       -0.01447387, -0.00183222,  0.01700537, -0.01465707,  0.00246549,
       -0.00685241, -0.01066092, -0.01299104, -0.00723331,  0.00789676,
        0.00832701,  0.00766651,  0.00268007,  0.00156526, -0.01550041,
        0.00623819, -0.00766718,  0.01275081, -0.01190558, -0.0091341 ,
       -0.0049164 ,  0.00138308, -0.01293811,  0.00163409, -0.00230528,
        0.01075449,  0.00939841, -0.00132029, -0.02348084, -0.00132714,
       -0.0001678 ,  0.00799193,  0.00372798,  0.00546762, -0.00259798,
       -0.0072659 , -0.00369504,  0.00296354,  0.00107311,  0.01544557,
        0.01017889,  0.00636788, -0.00963001,  0.00619133,  0.00781801,
       -0.0025125 , -0.01135393,  0.00594966, -0.00616527, -0.0185001 ,
       -0.00508632,  0.00110743, -0.00171944, -0.00469765,  0.01442408,
        0.01139572,  0.0111323 ,  0.00843756,  0.00738503,  0.00110754,
        0.01737408, -0.00158948, -0.00112388, -0.00071446,  0.00

In [58]:
scores,idx = getMostSimilarWords(E,E[W2I['bad'],:],W2I)
for i in range(1,len(idx)):
    print(scores[i],I2W[idx[i]])

0.4395656 cpcs
0.42632574 destroy
0.3979377 gallows
0.39327708 inscription
0.3923432 popov
0.39206317 pang
0.38972038 cassatt
0.3870611 during
0.38628685 psychoanalyst


In [59]:
def analogy(E,W2I,man,woman,king):
    v = E[W2I[king],:]-E[W2I[man],:]+E[W2I[woman],:]
    scores,idx = getMostSimilarWords(E,v,W2I)
    return scores,idx

In [60]:
scores,idx = analogy(E,W2I,'man','woman','king')
for i in range(len(idx)):
    print(scores[i],I2W[idx[i]])

0.5602565 woman
0.4060841 unction
0.3969176 ennis
0.3880951 goats
0.36448717 diviners
0.3629533 chara
0.36115432 substitutes
0.36074463 aldehyde
0.3580872 potentials
0.35731122 bauxite


In [61]:
from sklearn.decomposition import PCA

In [62]:
def reduceDim(X,new_dim=3):
    pca = PCA(n_components=new_dim)
    pca.fit(X)
    X_new = pca.transform(X)
    return X_new

In [63]:
X = np.random.rand(50,100)
X.shape

(50, 100)

In [64]:
X_new = reduceDim(X)
X_new.shape

(50, 3)

In [65]:
def sampleWords(E,W2I,wordsToSample):
    indx = [W2I[word] for word in wordsToSample]
    E_sampled = E[indx,:]
    return E_sampled

In [66]:
wordsToSample = ['coffee', 'tea', 'beer', 'wine', 'brandy', 'rum', 'champagne', 'water',
                         'spaghetti', 'borscht', 'hamburger', 'pizza', 'falafel', 'sushi', 'meatballs',
                         'dog', 'horse', 'cat', 'monkey', 'parrot', 'koala', 'lizard',
                         'frog', 'toad', 'monkey', 'ape', 'kangaroo', 'wombat', 'wolf',
                         'france', 'germany', 'hungary', 'luxembourg', 'australia', 'fiji', 'china',
                         'homework', 'assignment', 'problem', 'exam', 'test', 'class',
                         'school', 'college', 'university', 'institute']

In [68]:
def sampleWords_safe(E, W2I, wordsToSample):
    idx, missing = [], []
    for w in wordsToSample:
        k = w.lower()
        if k in W2I:
            idx.append(W2I[k])
        else:
            missing.append(w)
    if not idx:
        raise ValueError(f"No words found in vocab. OOV={missing}")
    return E[idx, :], missing

# Use the safer version
E_sampled, oov = sampleWords_safe(E, W2I, wordsToSample)

print("Sampling completed successfully!")
print("OOV (missing) words:", oov)

Sampling completed successfully!
OOV (missing) words: ['falafel', 'meatballs']


In [70]:
E_sampled.shape

(44, 100)

In [71]:
E_sampled_new = reduceDim(E_sampled)

In [72]:
E_sampled_new.shape

(44, 3)

In [73]:
import plotly.express as px
import pandas as pd

In [74]:
def plot3D(X,Y,Z,labels):
    df = pd.DataFrame({'X':X,'Y':Y,'Z':Z,'C':labels})
    fig = px.scatter_3d(df,x='X',y='Y',z='Z',color='C')
    fig.show()

In [75]:
plot3D(E_sampled_new[:,0],E_sampled_new[:,1],E_sampled_new[:,2],wordsToSample)

ValueError: All arrays must be of the same length